In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import sys
import joblib
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import plotly.io as pio
pio.renderers.default = 'png'
sys.path.append('..')

In [2]:
def apply_pca(X_scaled, n_components=None, variance_threshold=0.50):
    """
    Apply PCA to the scaled features.
    
    Parameters:
    - X_scaled: Scaled feature matrix
    - n_components: Number of components to keep (if None, use variance_threshold)
    - variance_threshold: Threshold of explained variance ratio to determine components
    
    Returns:
    - X_pca: Transformed features
    - pca: Fitted PCA object
    """
    if n_components is None:
        # Initialize PCA without specifying number of components
        pca = PCA()
        pca.fit(X_scaled)
        
        # Determine number of components based on explained variance
        cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
        n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Selected {n_components} components to explain {variance_threshold*100:.1f}% of variance")
        
        # Refit PCA with the determined number of components
        pca = PCA(n_components=n_components)
    else:
        pca = PCA(n_components=n_components)
    
    # Transform the data
    X_pca = pca.fit_transform(X_scaled)
    
    return X_pca, pca

In [3]:
def plot_explained_variance_plotly(pca):
    """
    Plot the explained variance ratio of PCA components using Plotly.
    """
    # Create data for the plot
    components = list(range(1, len(pca.explained_variance_ratio_) + 1))
    variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(variance_ratio)
    
    # Create a figure with secondary y-axis
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Add bar chart for individual explained variance
    fig.add_trace(
        go.Bar(x=components, y=variance_ratio, name="Explained Variance"),
        secondary_y=False,
    )
    
    # Add line chart for cumulative explained variance
    fig.add_trace(
        go.Scatter(x=components, y=cumulative_variance, name="Cumulative Variance", line=dict(color='red')),
        secondary_y=True,
    )
    
    # Set titles and labels
    fig.update_layout(
        title_text="Explained Variance by Principal Component",
        xaxis_title="Principal Component",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # Set y-axes titles
    fig.update_yaxes(title_text="Explained Variance Ratio", secondary_y=False)
    fig.update_yaxes(title_text="Cumulative Explained Variance", secondary_y=True)
    
    # Add horizontal line at common variance thresholds
    for threshold in [0.25, 0.5, 0.75, 0.95]:
        fig.add_hline(y=threshold, line=dict(color="green", width=1, dash="dash"), 
                     annotation_text=f"{threshold*100}%", annotation_position="right",
                     secondary_y=True)
    
    return fig

In [4]:
def plot_feature_contributions_plotly(pca, feature_names, n_top_features=10):
    """
    Plot the feature contributions to principal components using Plotly.
    """
    # Create figures for top components
    n_components = min(2, pca.n_components_)
    figures = []
    
    for i in range(n_components):
        component = pca.components_[i]
        
        # Get indices of top contributing features (by absolute value)
        top_indices = np.argsort(np.abs(component))[::-1][:n_top_features]
        
        # Extract feature names and contribution values
        features = [feature_names[idx] for idx in top_indices]
        contributions = component[top_indices]
        
        # Create horizontal bar chart
        fig = go.Figure()
        
        # Add bars colored by contribution direction (positive/negative)
        colors = ['blue' if x >= 0 else 'red' for x in contributions]
        
        fig.add_trace(go.Bar(
            y=features,
            x=contributions,
            orientation='h',
            marker_color=colors
        ))
        
        # Set titles and labels
        fig.update_layout(
            title=f"Top Features Contributing to Principal Component {i+1}",
            xaxis_title="Component Coefficient",
            yaxis_title="Feature",
            yaxis=dict(autorange="reversed"),  # Reverse y-axis to show highest values at top
            height=500
        )
        
        figures.append(fig)
    
    return figures

In [5]:
def plot_pca_2d_plotly(X_pca, y, pca, feature_names):
    """
    Plot the first two PCA components with class labels and feature vectors using Plotly.
    
    Parameters:
    - X_pca: PCA transformed data
    - y: Target labels
    - pca: Fitted PCA object
    - feature_names: List of original feature names
    """
    if X_pca.shape[1] < 2:
        print("Not enough components for 2D visualization")
        return None
    
    # Create a DataFrame for plotting
    df_plot = pd.DataFrame({
        'PC1': X_pca[:, 0],
        'PC2': X_pca[:, 1],
        'Class': y
    })
    
    # Map class labels to meaningful names
    df_plot['Class_Label'] = df_plot['Class'].map({0: 'Legitimate', 1: 'Phishing'})
    
    # Create scatter plot for data points
    fig = px.scatter(
        df_plot, 
        x='PC1', 
        y='PC2', 
        color='Class_Label',
        color_discrete_map={'Legitimate': 'blue', 'Phishing': 'red'},
        opacity=0.7,
        title="PCA Biplot: First Two Principal Components",
        labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'},
        hover_data=['Class_Label']
    )
    
    # Add feature vectors
    # Scale the feature vectors for better visualization
    scale_factor = 0.5 * min(
        np.ptp(X_pca[:, 0]),  # range of PC1
        np.ptp(X_pca[:, 1])   # range of PC2
    )
    
    # Get top 10 features by contribution magnitude
    top_features = 10
    feature_contributions = np.sqrt(
        pca.components_[0]**2 + pca.components_[1]**2
    )
    top_indices = np.argsort(feature_contributions)[-top_features:]
    
    # Add arrows for each top feature
    for idx in top_indices:
        x = pca.components_[0, idx] * scale_factor
        y = pca.components_[1, idx] * scale_factor
        
        fig.add_annotation(
            x=x, y=y,
            ax=0, ay=0,
            xref="x", yref="y",
            axref="x", ayref="y",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="black"
        )
        
        # Add feature name label
        fig.add_annotation(
            x=x, y=y,
            text=feature_names[idx],
            showarrow=False,
            font=dict(size=10),
            xanchor="left" if x > 0 else "right",
            yanchor="bottom"
        )
    
    # Update layout
    fig.update_layout(
        legend_title="Class",
        height=600, 
        width=800,
        showlegend=True
    )
    
    return fig

In [6]:
def train_logistic_regression(X_train, X_test, y_train, y_test):
    """
    Train a logistic regression model on PCA-transformed data.
    
    Returns:
    - model: Trained logistic regression model
    - performance: Dictionary with performance metrics
    """
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    performance = {
        'classification_report': classification_report(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }
    
    return model, performance

In [7]:
def plot_roc_curve_plotly(y_test, y_pred_proba):
    """
    Plot the ROC curve for the logistic regression model using Plotly.
    """
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Create figure
    fig = go.Figure()
    
    # Add ROC curve
    fig.add_trace(go.Scatter(
        x=fpr, 
        y=tpr,
        mode='lines',
        name=f'ROC Curve (AUC = {auc:.3f})',
        line=dict(color='blue', width=2)
    ))
    
    # Add random classifier line
    fig.add_trace(go.Scatter(
        x=[0, 1], 
        y=[0, 1],
        mode='lines',
        name='Random Classifier',
        line=dict(color='gray', width=2, dash='dash')
    ))
    
    # Update layout
    fig.update_layout(
        title='ROC Curve - Logistic Regression on PCA Features',
        xaxis_title='False Positive Rate',
        yaxis_title='True Positive Rate',
        legend=dict(x=0.7, y=0.1),
        width=700,
        height=500,
        xaxis=dict(range=[0, 1], constrain='domain'),
        yaxis=dict(range=[0, 1], constrain='domain'),
        showlegend=True
    )
    
    return fig

In [8]:
def plot_confusion_matrix_plotly(confusion_mat):
    """
    Plot the confusion matrix using Plotly.
    """
    # Define labels
    labels = ['Legitimate', 'Phishing']
    
    # Create annotation text
    annotations = []
    for i in range(len(confusion_mat)):
        for j in range(len(confusion_mat[i])):
            annotations.append(
                dict(
                    x=j,
                    y=i,
                    text=str(confusion_mat[i, j]),
                    font=dict(color='white' if confusion_mat[i, j] > confusion_mat.max()/2 else 'black', size=14),
                    showarrow=False
                )
            )
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=confusion_mat,
        x=labels,
        y=labels,
        colorscale='Blues',
    ))
    
    # Add annotations
    fig.update_layout(
        title='Confusion Matrix - Logistic Regression on PCA Features',
        xaxis_title='Predicted Label',
        yaxis_title='True Label',
        xaxis=dict(side='bottom'),
        annotations=annotations,
        width=600,
        height=500
    )
    
    return fig 

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

def load_and_preprocess_data(file_path):
    """
    Load and preprocess the phishing dataset for machine learning.
    """
    # Load the data
    df = pd.read_csv(file_path)
    
    # Separate features and target
    X = df.drop(['label', 'FILENAME', 'URL', 'Domain', 'Title'], axis=1)
    y = df['label']
    
    # Convert boolean columns to int
    boolean_columns = X.select_dtypes(include=['bool']).columns
    X[boolean_columns] = X[boolean_columns].astype(int)
    
    # Handle categorical columns
    categorical_columns = X.select_dtypes(include=['object']).columns
    for col in categorical_columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Scale numerical features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Convert back to DataFrame to maintain column names
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    
    return X_scaled, y

def prepare_inference_data(file_path, scaler=None, label_encoders=None):
    """
    Prepare new data for inference using the same preprocessing steps.
    """
    # Load the data
    df = pd.read_csv(file_path)
    
    # Separate features
    X = df.drop(['FILENAME', 'URL', 'Domain', 'Title'], axis=1, errors='ignore')
    
    # Convert boolean columns to int
    boolean_columns = X.select_dtypes(include=['bool']).columns
    X[boolean_columns] = X[boolean_columns].astype(int)
    
    # Handle categorical columns
    categorical_columns = X.select_dtypes(include=['object']).columns
    for col in categorical_columns:
        if label_encoders and col in label_encoders:
            X[col] = label_encoders[col].transform(X[col].astype(str))
        else:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            if label_encoders is not None:
                label_encoders[col] = le
    
    # Scale numerical features
    if scaler:
        X_scaled = scaler.transform(X)
    else:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    
    # Convert back to DataFrame
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    
    return X_scaled, scaler, label_encoders

def save_preprocessing_artifacts(scaler, label_encoders, output_dir='artifacts'):
    """
    Save preprocessing artifacts for later use in inference.
    """
    import joblib
    import os
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Save scaler
    joblib.dump(scaler, os.path.join(output_dir, 'scaler.joblib'))
    
    # Save label encoders
    for col, le in label_encoders.items():
        joblib.dump(le, os.path.join(output_dir, f'label_encoder_{col}.joblib'))

def load_preprocessing_artifacts(input_dir='artifacts'):
    """
    Load preprocessing artifacts for inference.
    """
    import joblib
    import os
    
    # Load scaler
    scaler = joblib.load(os.path.join(input_dir, 'scaler.joblib'))
    
    # Load label encoders
    label_encoders = {}
    for file in os.listdir(input_dir):
        if file.startswith('label_encoder_'):
            col = file.replace('label_encoder_', '').replace('.joblib', '')
            label_encoders[col] = joblib.load(os.path.join(input_dir, file))
    
    return scaler, label_encoders

In [10]:
# Create output directories
os.makedirs('../plots', exist_ok=True)
os.makedirs('../pca_results', exist_ok=True)
os.makedirs('../artifacts', exist_ok=True)

# Load and preprocess data
file_path = '../../Data/Raw/PhiUSIIL_Phishing_URL_Dataset.csv'
X_scaled, y = load_and_preprocess_data(file_path)

# Save preprocessing artifacts for later use
# Create a scaler and fit it to the data
scaler = StandardScaler()
scaler.fit(X_scaled)

# Create empty label encoders dictionary
label_encoders = {}

# Read the original data to get categorical columns
df = pd.read_csv(file_path)
X = df.drop(['label', 'FILENAME', 'URL', 'Domain', 'Title'], axis=1)
categorical_columns = X.select_dtypes(include=['object']).columns

# Create label encoders for each categorical column
for col in categorical_columns:
    le = LabelEncoder()
    le.fit(X[col].astype(str))
    label_encoders[col] = le

# Save the preprocessing artifacts
save_preprocessing_artifacts(scaler, label_encoders)

### 4.1 Análisis de componentes principales PCA.

Cargamos y pre-procesamos los datos.  PCA necesita variables con media cero y varianza unitaria.  Tiramos las columnas 'titulo, url, domain' porque básicamente sus valores son únicos y lo importante de la URL ya está capturado en las otras features.

In [11]:
# Apply PCA (automatically determine number of components)
variance_threshold = 0.75  # Explain 75% of variance
X_pca, pca = apply_pca(X_scaled, variance_threshold=variance_threshold)

print("PCA transformed data shape:", X_pca.shape)
print(f"Explained variance ratio: {pca.explained_variance_ratio_[:10]}")

Selected 20 components to explain 75.0% of variance
PCA transformed data shape: (235795, 20)
Explained variance ratio: [0.17995189 0.09808057 0.05412036 0.04345818 0.03675129 0.03558306
 0.03126925 0.02930828 0.02716744 0.02425956]


Se requieren 20 componentes para llegar a 75% de varianza explicada.
Sin embargo se puede ver un quiebre del segundo al tercer componente.
Por la regla del codo (imagen no mostrada) y por simplicidad intentemos trabajar con 2 componentes principales.

### Descripción e interpretación de los componenes principales.

In [12]:
# Plot and analyze principal components
n_components = 2  # Since we're focusing on first 2 PCs

# Get feature contributions for each component
feature_contributions = pd.DataFrame(
    pca.components_[:n_components],
    columns=X_scaled.columns
)

# Get top 8 features for each component
top_features = []
for i in range(n_components):
    pc_contributions = feature_contributions.iloc[i].abs().sort_values(ascending=False)
    top_features.append(pc_contributions.head(8).index.tolist())

# Create HTML table with features as rows and components as columns
html_output = "<table style='border-collapse: collapse; width: 50%; margin: 0 auto;'>"  # Changed width to 50% and centered
html_output += "<tr><th colspan='3' style='border: 1px solid black; padding: 4px; text-align: center; font-size: 0.9em;'>Principal Component Analysis</th></tr>"
html_output += "<tr><th style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>Feature</th>"
html_output += "<th style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>PC1</th>"
html_output += "<th style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>PC2</th></tr>"

# Add rows for each feature
for i in range(8):
    html_output += "<tr>"
    html_output += f"<td style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>Feature {i+1}</td>"
    html_output += f"<td style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>{top_features[0][i]}</td>"
    html_output += f"<td style='border: 1px solid black; padding: 4px; font-size: 0.9em;'>{top_features[1][i]}</td>"
    html_output += "</tr>"

html_output += "</table>"

# Display the HTML table
from IPython.display import HTML
display(HTML(html_output))

### Interpretación de los factores principales
El factor 1 tiene que ver con legitimidad de la página,
qué tan similar es a páginas legítimas, y si posee otros marcadores de autenticidad
de ser auténtica como CopyRightInfo, si tiene links a redes sociales, similitud a páginas legítimas.
Cuanto mayor tiene caracteres especiales en la URL, menor es legítima.

El factor 2 es básicamente el análisis léxico estático de la URL para ofuscación.
captura la intención maliciosa del atacante al alterar la URL al incluir muchos caracteres, muchos digitos, signos 
de redirección, de query params, botones para mandar forms. 

### El componente 1 es la DEFENSA del phishing, porque son marcadores de legitimidad.
### El componente 2 es el ATAQUE del phishing, son marcadores de malicia.

In [16]:
# Plot PCA 2D visualization with smaller dimensions
fig_2d = plot_pca_2d_plotly(X_pca, y, pca, feature_names=X.columns)
fig_2d.update_layout(
    width=600,  # Smaller width
    height=400,  # Smaller height
    xaxis=dict(range=[-30, 30]),
    yaxis=dict(range=[-20, 50])
)

fig_2d.show()

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


Notamos que los datos están bien separados en las direcciones principales.
Lo que significa que hemos capturado una parte de la esencia del phishing con estas dos componentes
Sin embargo se empalman muy cerca del origen lo cual quiere decir que hay muestras indistinguibles usando solamente las 2 primera componentes.


### 2. Ahora entrenamos un modelo de regresión logística
Usando X_PCA con dos componentes, este es un caso de clasificación en donde las clases no están balanceadas Además de eso, error T2 (decir que es benigno cuando es phishing) puede ser muy costoso para el usuario.  Por lo que una métrica como accuracy no es buena.  Podemos usar F1 que balancea precisión y recall.  Y también veremos área bajo la curva de ROC.

In [17]:
# Save PCA results to CSV
pca_df = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'])
pca_df['Ground_Truth'] = y.values
pca_df['URL'] = df['URL'].values

# Save to CSV
pca_df.to_csv('../pca_results/pca_2d_results.csv', index=False)
print(f"Saved PCA results to CSV with shape: {pca_df.shape}")


Saved PCA results to CSV with shape: (235795, 4)


In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score
# Split data for training
X_train, X_test, y_train, y_test = train_test_split(
    X_pca[:, :2], y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Get predictions
y_pred = model.predict(X_test)

# Calculate metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print performance metrics
print("\nModel Performance Report:")
print("-" * 50)
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")



Model Performance Report:
--------------------------------------------------
Precision: 0.983
Recall: 0.980
F1 Score: 0.982


Vemos que nuestro predictor tiene >95% de eficiacia en todas las métricas relevantes. Lo cual es poco usual para datasets 

### En conclusión, nuestro predictor es eficaz para clasificar URLs y detectar el phishing.
### Las direcciones principales **capturan la esencia del phishing** y se le da una interpretación al problema en términos de ciberseguridad. 
### Si hicieramos un software como una extensión de chrome para detectar phishing, podríamos tener éxito comercial.